# CNN-GRU : comparaison de deux codages des cibles

Ce notebook entraîne un CNN-GRU, l'architecture utilisée dans l'article de référence, avec deux codages des cibles, dans les mêmes conditions, puis compare leurs performances en unités physiques :
- **codage contraint** : HLC_mit, HLC_ext, log τ_n, log(τ_∞ − τ_n), log(τ_b − τ_∞), qui garantit τ_n < τ_∞ < τ_b ;
- **codage direct** : HLC_mit, HLC_ext, τ_b, τ_n, τ_∞, dont l'ordre est vérifié après prédiction.

Dans les deux cas, HLC = HLC_mit + HLC_ext est garanti par construction. Les données préparées et le découpage (`splits.pt`) sont relus tels quels.

## 1. Configuration et chargement des données

In [ ]:
import copy
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset

# --- Chemins (Kaggle) ---
DATA = Path("/kaggle/input/datasets/jiihan/coenergy")   # données préparées + splits.pt + .py
OUT = Path("/kaggle/working")                           # modèles et historiques sauvegardés

# --- Réglages communs aux deux expériences ---
SEED = 42
N_EPOCHS = 10
LR = 5e-4
BATCH_SIZE = 256
N_TRAIN = None          # None = tout le train (800 000) ; ex. 200_000 pour un essai plus rapide
P_MAX = 7000.0

sys.path.append(str(DATA))
from data_utils import load_data

T, U, Y, targets = load_data(DATA)          # T en °C, U en W (float32), Y (N, 6)
SPLITS = torch.load(DATA / "splits.pt")     # découpage stratifié déjà calculé

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Appareil :", DEVICE, torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "")
print(T.shape, U.shape, Y.shape)

In [ ]:
targets = [str(t) for t in targets]
col = {name: i for i, name in enumerate(targets)}
N, L = T.shape

NO_EXC = U.sum(1) == 0                      # simulations sans excitation
ALWAYS_ON = (U == P_MAX).all(1)             # chauffage permanent

for name, idx in SPLITS.items():
    print(f"{name:6s} {len(idx):7d} simulations | sans excitation {NO_EXC[idx].float().mean().item():.2%}"
          f" | toujours allumé {ALWAYS_ON[idx].float().mean().item():.2%}")

## 2. Normalisation des entrées

In [ ]:
# Une seule moyenne et un seul écart-type pour T, calculés sur le train :
# l'amplitude de la réponse en température porte l'information sur le HLC.
T_SD, T_MU = torch.std_mean(T[SPLITS["train"][:100_000]])
T_MU, T_SD = T_MU.item(), T_SD.item()
print(f"T (100 000 simulations du train) : moyenne = {T_MU:.3f} °C, écart-type = {T_SD:.3f} °C")

## 3. Les deux codages des cibles

Chaque codage est une paire de fonctions : `encode` transforme les 6 indicateurs en 5 valeurs, `decode` fait l'inverse. Les 5 valeurs sont ensuite standardisées avec les statistiques du train.

In [ ]:
def encode_contraint(y: torch.Tensor) -> torch.Tensor:
    """6 indicateurs -> HLC_mit, HLC_ext, log tau_n, log(tau_inf - tau_n), log(tau_b - tau_inf)."""
    tb, tn, ti = y[:, col["tau_b"]], y[:, col["tau_n"]], y[:, col["tau_inf"]]
    return torch.stack([y[:, col["HLC_mit"]], y[:, col["HLC_ext"]],
                        tn.log(), (ti - tn).log(), (tb - ti).log()], dim=1)


def decode_contraint(z: torch.Tensor) -> torch.Tensor:
    """5 valeurs -> 6 indicateurs : HLC = somme, et tau_n < tau_inf < tau_b garantis."""
    tn = z[:, 2].exp()
    ti = tn + z[:, 3].exp()
    tb = ti + z[:, 4].exp()
    y = torch.empty(len(z), 6, dtype=z.dtype)
    y[:, col["HLC_mit"]], y[:, col["HLC_ext"]] = z[:, 0], z[:, 1]
    y[:, col["HLC"]] = z[:, 0] + z[:, 1]
    y[:, col["tau_n"]], y[:, col["tau_inf"]], y[:, col["tau_b"]] = tn, ti, tb
    return y


def encode_direct(y: torch.Tensor) -> torch.Tensor:
    """6 indicateurs -> HLC_mit, HLC_ext, tau_b, tau_n, tau_inf."""
    return torch.stack([y[:, col["HLC_mit"]], y[:, col["HLC_ext"]],
                        y[:, col["tau_b"]], y[:, col["tau_n"]], y[:, col["tau_inf"]]], dim=1)


def decode_direct(z: torch.Tensor) -> torch.Tensor:
    """5 valeurs -> 6 indicateurs : HLC = somme garanti, ordre des tau non garanti."""
    y = torch.empty(len(z), 6, dtype=z.dtype)
    y[:, col["HLC_mit"]], y[:, col["HLC_ext"]] = z[:, 0], z[:, 1]
    y[:, col["HLC"]] = z[:, 0] + z[:, 1]
    y[:, col["tau_b"]], y[:, col["tau_n"]], y[:, col["tau_inf"]] = z[:, 2], z[:, 3], z[:, 4]
    return y


ENCODINGS = {"contraint": (encode_contraint, decode_contraint),
             "direct": (encode_direct, decode_direct)}

for name, (enc, dec) in ENCODINGS.items():
    y = Y[:1000].double()
    assert torch.allclose(dec(enc(y)), y, rtol=1e-6), name
    print(f"Codage {name} : aller-retour vérifié.")

## 4. Dataset et métriques

In [ ]:
class MyDataset(Dataset):
    """Renvoie une simulation : entrée (2, L) = [T normalisé, U / P_MAX] et cible standardisée (5,)."""

    def __init__(self, idx: torch.Tensor, z_std: torch.Tensor, noise_c: float = 0.0) -> None:
        self.idx = idx              # numéros des simulations de cet ensemble
        self.z_std = z_std          # cibles standardisées (N, 5) du codage utilisé
        self.noise_c = noise_c      # bruit de capteur (°C), 0 = aucun

    def __len__(self) -> int:
        return len(self.idx)

    def __getitem__(self, i: int) -> tuple[torch.Tensor, torch.Tensor]:
        sim = self.idx[i]
        t = T[sim]
        if self.noise_c > 0:
            t = t + self.noise_c * torch.randn_like(t)
        x = torch.stack([(t - T_MU) / T_SD, U[sim] / P_MAX])
        return x, self.z_std[sim]


def regression_metrics(y_true: torch.Tensor, y_pred: torch.Tensor) -> dict:
    """RMSE, MAE, P95, MAPE et R² par indicateur, en unités physiques."""
    out = {}
    for name in targets:
        t, p = y_true[:, col[name]].double(), y_pred[:, col[name]].double()
        err = p - t
        out[name] = {"RMSE": err.pow(2).mean().sqrt().item(),
                     "MAE": err.abs().mean().item(),
                     "P95": err.abs().quantile(0.95).item(),
                     "MAPE": 100 * (err / t).abs().mean().item(),
                     "R2": (1 - err.pow(2).sum() / (t - t.mean()).pow(2).sum()).item()}
    return out


RESULTS = {}


def evaluate(name: str, y_pred: torch.Tensor, split: str = "val") -> None:
    """Évalue un modèle globalement, avec et sans excitation, et enregistre le résultat."""
    idx = SPLITS[split]
    y_true, no_exc = Y[idx], NO_EXC[idx]
    RESULTS[name] = {g: regression_metrics(y_true[m], y_pred[m])
                     for g, m in [("tout", torch.ones_like(no_exc)),
                                  ("avec excitation", ~no_exc), ("sans excitation", no_exc)]}
    print(f"=== {name} ({split}) ===")
    print(f"{'':12s}{'RMSE':>9s}{'MAE':>9s}{'P95':>9s}{'MAPE %':>9s}{'R²':>8s}{'R² exc.':>9s}")
    for t in targets:
        m, me = RESULTS[name]["tout"][t], RESULTS[name]["avec excitation"][t]
        print(f"{t:12s}{m['RMSE']:9.2f}{m['MAE']:9.2f}{m['P95']:9.2f}"
              f"{m['MAPE']:9.2f}{m['R2']:8.3f}{me['R2']:9.3f}")


# Baseline : prédire la moyenne du train
evaluate("Moyenne", Y[SPLITS["train"]].mean(0).expand(len(SPLITS["val"]), -1))

## 5. Le modèle CNN-GRU

L'article de référence utilise un CNN-GRU sans en détailler l'architecture. Celle retenue ici :
- **3 blocs convolutifs** (Conv1d noyau 5 → BatchNorm → GELU → MaxPool ×2) : ils extraient des motifs locaux (montées et descentes après chaque créneau) et réduisent la séquence de 480 à 60 pas, chaque pas résumant 8 minutes ;
- **un GRU bidirectionnel à 2 couches**, 64 unités par direction : il parcourt la séquence réduite et accumule une mémoire de la dynamique, à l'image de l'état thermique du bâtiment. La lecture dans les deux sens est possible car toute la séquence est observée avant de prédire ;
- **les états finaux des deux directions** (128 valeurs) passent dans la même tête MLP que le TCN.

In [ ]:
class CNNGRURegressor(nn.Module):
    """CNN-GRU : séquence (B, 2, L) -> 5 sorties standardisées."""

    def __init__(self, in_channels: int = 2, channels: int = 64, kernel: int = 5, n_conv: int = 3,
                 hidden: int = 64, n_gru_layers: int = 2, dropout: float = 0.1, n_out: int = 5) -> None:
        super().__init__()
        layers, c_in = [], in_channels
        for _ in range(n_conv):
            layers += [nn.Conv1d(c_in, channels, kernel, padding=kernel // 2),
                       nn.BatchNorm1d(channels), nn.GELU(), nn.MaxPool1d(2)]
            c_in = channels
        self.cnn = nn.Sequential(*layers)                      # (B, channels, L / 2^n_conv)
        self.gru = nn.GRU(input_size=channels, hidden_size=hidden, num_layers=n_gru_layers,
                          batch_first=True, bidirectional=True, dropout=dropout)
        self.head = nn.Sequential(nn.Linear(2 * hidden, 64), nn.GELU(),
                                  nn.Dropout(dropout), nn.Linear(64, n_out))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        h = self.cnn(x)                      # (B, channels, 60)
        h = h.transpose(1, 2)                # (B, 60, channels) : le GRU attend (batch, temps, canaux)
        _, h_n = self.gru(h)                 # h_n : (n_couches * 2, B, hidden)
        h = torch.cat([h_n[-2], h_n[-1]], dim=1)   # états finaux de la dernière couche, aller et retour
        return self.head(h)


def weights_init(layer_in):
    """Initialisation de He pour les convolutions et les couches linéaires (le GRU garde la sienne)."""
    if isinstance(layer_in, (nn.Linear, nn.Conv1d)):
        nn.init.kaiming_uniform_(layer_in.weight)
        layer_in.bias.data.fill_(0.0)


m = CNNGRURegressor()
print(f"Paramètres : {sum(p.numel() for p in m.parameters()):,}")
with torch.no_grad():
    print("Forme après le CNN :", tuple(m.cnn(torch.zeros(2, 2, L)).shape), "| sortie :", tuple(m(torch.zeros(2, 2, L)).shape))
del m

## 6. Entraînement

Une seule fonction lance une expérience complète pour un codage donné : standardisation des cibles, loaders, création d'un **nouveau** CNN-GRU, boucle d'entraînement, sélection du meilleur modèle sur la validation, prédiction et évaluation. Les deux expériences utilisent ainsi exactement les mêmes réglages et la même graine.

In [ ]:
@torch.no_grad()
def predict(model, loader, z_mu, z_sd, decode) -> torch.Tensor:
    """Prédictions en unités physiques (n, 6). Le loader ne doit PAS être mélangé."""
    model.eval()
    z = torch.cat([model(x.to(DEVICE)).float().cpu() for x, _ in loader])
    return decode((z * z_sd + z_mu).double()).float()


def run_experiment(encoding: str) -> dict:
    """Entraîne et évalue un CNN-GRU avec le codage des cibles donné."""
    torch.manual_seed(SEED)
    encode, decode = ENCODINGS[encoding]

    # Cibles standardisées pour ce codage (statistiques du train)
    Z = encode(Y.double()).float()
    z_mu, z_sd = Z[SPLITS["train"]].mean(0), Z[SPLITS["train"]].std(0)
    z_std = (Z - z_mu) / z_sd

    train_idx = SPLITS["train"] if N_TRAIN is None else SPLITS["train"][:N_TRAIN]
    train_loader = DataLoader(MyDataset(train_idx, z_std), batch_size=BATCH_SIZE, shuffle=True,
                              drop_last=True, num_workers=2, pin_memory=True)
    val_loader = DataLoader(MyDataset(SPLITS["val"], z_std), batch_size=1024, shuffle=False,
                            num_workers=2, pin_memory=True)

    # Nouveau modèle, initialisation, perte, optimiseur, scheduler
    model = CNNGRURegressor()
    model.apply(weights_init)
    model.to(DEVICE)
    criterion = nn.MSELoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.5)

    best_val, best_state, history = float("inf"), None, []
    for epoch in range(N_EPOCHS):
        model.train()
        epoch_loss = 0.0
        for x_batch, y_batch in train_loader:
            x_batch, y_batch = x_batch.to(DEVICE), y_batch.to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(model(x_batch), y_batch)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            epoch_loss += loss.item()
        epoch_loss /= len(train_loader)

        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for x_batch, y_batch in val_loader:
                val_loss += criterion(model(x_batch.to(DEVICE)), y_batch.to(DEVICE)).item()
        val_loss /= len(val_loader)

        print(f"[{encoding}] Epoch {epoch:3d}, train loss {epoch_loss:.4f}, val loss {val_loss:.4f}, "
              f"lr {scheduler.get_last_lr()[0]:.1e}")
        history.append({"epoch": epoch + 1, "train": epoch_loss, "val": val_loss})
        torch.save(history, OUT / f"cnngru_{encoding}_history.pt")

        if val_loss < best_val:
            best_val = val_loss
            best_state = copy.deepcopy(model.state_dict())
            torch.save(best_state, OUT / f"cnngru_{encoding}_best.pt")
        scheduler.step()

    model.load_state_dict(best_state)
    y_pred = predict(model, val_loader, z_mu, z_sd, decode)
    evaluate(f"CNN-GRU {encoding}", y_pred)
    return {"model": model, "history": history, "y_pred": y_pred}

### 6.1 Codage contraint

In [ ]:
run_contraint = run_experiment("contraint")

### 6.2 Codage direct

In [ ]:
run_direct = run_experiment("direct")

## 7. Comparaison

In [ ]:
# Courbes d'apprentissage (attention : les pertes portent sur des cibles différentes,
# elles montrent la dynamique d'apprentissage mais ne se comparent pas en valeur absolue)
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5), sharex=True)
for ax, (name, run) in zip(axes, [("contraint", run_contraint), ("direct", run_direct)]):
    ep = [h["epoch"] for h in run["history"]]
    ax.plot(ep, [h["train"] for h in run["history"]], marker="o", label="entraînement")
    ax.plot(ep, [h["val"] for h in run["history"]], marker="o", label="validation")
    ax.set(title=f"CNN-GRU — codage {name}", xlabel="Époque", ylabel="MSE (cibles standardisées)")
    ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# R² par indicateur, en unités physiques (validation)
models = ["Moyenne", "CNN-GRU contraint", "CNN-GRU direct"]
for group in ["tout", "avec excitation", "sans excitation"]:
    print(f"\nR² — {group}")
    print(f"{'':20s}" + "".join(f"{t:>10s}" for t in targets))
    for name in models:
        print(f"{name:20s}" + "".join(f"{RESULTS[name][group][t]['R2']:10.3f}" for t in targets))

# Erreur moyenne absolue, en unités physiques
print(f"\nMAE (W/K pour les HLC, h pour les tau) — tout")
print(f"{'':20s}" + "".join(f"{t:>10s}" for t in targets))
for name in models:
    print(f"{name:20s}" + "".join(f"{RESULTS[name]['tout'][t]['MAE']:10.2f}" for t in targets))

In [ ]:
# Respect des contraintes physiques
for name, run in [("contraint", run_contraint), ("direct", run_direct)]:
    y = run["y_pred"]
    somme = torch.allclose(y[:, col["HLC"]], y[:, col["HLC_mit"]] + y[:, col["HLC_ext"]])
    ordre = (y[:, col["tau_n"]] < y[:, col["tau_inf"]]) & (y[:, col["tau_inf"]] < y[:, col["tau_b"]])
    print(f"Codage {name:9s} | HLC = HLC_mit + HLC_ext : {somme} | "
          f"violations de l'ordre des tau : {(~ordre).float().mean().item():.2%}")

In [ ]:
# Graphique de synthèse : R² par indicateur (simulations avec excitation)
x = np.arange(len(targets))
width = 0.38
fig, ax = plt.subplots(figsize=(11, 4.5))
for k, name in enumerate(["CNN-GRU contraint", "CNN-GRU direct"]):
    vals = [RESULTS[name]["avec excitation"][t]["R2"] for t in targets]
    bars = ax.bar(x + (k - 0.5) * width, vals, width, label=name)
    ax.bar_label(bars, fmt="%.2f", fontsize=8)
ax.set_xticks(x, targets)
ax.axhline(0, color="black", lw=0.8)
ax.set_ylabel("R² (validation, avec excitation)")
ax.set_title("Comparaison des deux codages des cibles")
ax.legend()
plt.tight_layout()
plt.show()